# 10_phase4_QC_sensitivity.ipynb
Phase 4 — QC threshold sensitivity analysis (GSE176078)

**Question:** does the choice of n_genes_by_counts threshold (baseline: >500) materially change downstream clustering and the headline findings?

**Design:** full pipeline re-run at two alternative thresholds — a looser one (>300, includes more, lower-quality cells) and a stricter one (>750, excludes more borderline cells) — same pipeline as the validated baseline (seeded PCA/Harmony/Leiden, CSR-forced, memory-safe chunking), for a fair comparison.

**What's checked:** total cells retained, cluster count at resolution 0.6, and whether the HER2+ Memory T cell composition finding (the headline scCODA result) still shows the same directional pattern (HER2+ > ER+ and HER2+ > TNBC).

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# NUMBA_NUM_THREADS must be set BEFORE scanpy is imported anywhere in
# this kernel — Numba locks in its thread count the first time it runs
# any computation, and won't accept changes afterward in the same session.
# ----------------------------
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import scrublet as scr
import matplotlib.pyplot as plt
from scipy.sparse import issparse, csr_matrix, vstack
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_sensitivity"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_sensitivity"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [3]:
# ----------------------------
# Cell 2 — Reusable pipeline function, parameterised by QC threshold
# Same validated pipeline as 03_phase1_QC_V2.ipynb: no dask, forced CSR,
# seeded PCA/Harmony, memory-safe chunking. Runs QC -> Scrublet ->
# normalise -> HVG -> scale -> PCA -> Harmony -> Leiden clustering.
# ----------------------------
def run_pipeline_at_threshold(n_genes_threshold, label):
    print(f"\n{'='*70}")
    print(f"RUNNING PIPELINE AT n_genes_by_counts > {n_genes_threshold} ({label})")
    print(f"{'='*70}\n")

    adata = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")
    if issparse(adata.X):
        adata.X = adata.X.tocsr().astype("float32", copy=False)
    else:
        adata.X = csr_matrix(adata.X, dtype=np.float32)
    gc.collect()

    # VDJ + constant region + MT gene removal (same as baseline)
    vdj_prefixes = ("IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
                     "IGHD", "IGHJ", "IGLJ", "IGKJ")
    constant_genes = ("TRAC", "TRBC", "TRGC", "TRDC", "IGHA", "IGHD", "IGHE",
                       "IGHG", "IGHM", "IGLC", "IGKC")
    all_prefixes = vdj_prefixes + constant_genes
    vdj_mask = ~adata.var_names.str.startswith(all_prefixes)
    mt_mask = ~adata.var_names.str.startswith("MT-")
    adata = adata[:, vdj_mask & mt_mask].copy()
    gc.collect()

    # QC metrics (chunked)
    n_cells = adata.n_obs
    n_genes_by_counts = np.zeros(n_cells, dtype=np.float32)
    total_counts = np.zeros(n_cells, dtype=np.float32)
    chunk_size = 2000
    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata.X[start:end]
        total_counts[start:end] = np.asarray(chunk.sum(axis=1)).flatten()
        n_genes_by_counts[start:end] = np.asarray((chunk > 0).sum(axis=1)).flatten()
    adata.obs["n_genes_by_counts"] = n_genes_by_counts
    adata.obs["total_counts"] = total_counts

    n_before = adata.n_obs
    adata = adata[adata.obs["n_genes_by_counts"] > n_genes_threshold].copy()
    sc.pp.filter_genes(adata, min_cells=3)
    n_after = adata.n_obs
    print(f"Cell filtering: {n_before} -> {n_after} cells "
          f"({n_before - n_after} removed, threshold n_genes>{n_genes_threshold})")
    gc.collect()

    # Scrublet per sample
    all_doublet_scores = np.zeros(adata.n_obs, dtype=np.float32)
    all_predicted_doublets = np.zeros(adata.n_obs, dtype=bool)
    samples = adata.obs["orig.ident"].unique()
    for sample in samples:
        sample_mask = adata.obs["orig.ident"] == sample
        sample_idx = np.where(sample_mask)[0]
        if len(sample_idx) < 50:
            continue
        chunks = []
        for start in range(0, len(sample_idx), chunk_size):
            end = min(start + chunk_size, len(sample_idx))
            global_idx = sample_idx[start:end]
            chunk = adata.X[global_idx]
            if not issparse(chunk): chunk = csr_matrix(chunk)
            chunks.append(chunk)
        X_sample = vstack(chunks)
        try:
            scrub = scr.Scrublet(X_sample)
            doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)
            all_doublet_scores[sample_idx] = doublet_scores
            all_predicted_doublets[sample_idx] = predicted_doublets
        except Exception as e:
            print(f"  Scrublet failed for {sample}: {e}")
        gc.collect()

    adata.obs["predicted_doublet"] = all_predicted_doublets
    before_doublets = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"]].copy()
    print(f"Doublets removed: {before_doublets - adata.n_obs} "
          f"(final: {adata.n_obs} cells)")
    gc.collect()

    # Normalise
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata

    # HVG, scale, PCA (seeded), Harmony (seeded)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    adata = adata[:, adata.var.highly_variable].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack", random_state=0)
    sce.pp.harmony_integrate(adata, key="orig.ident", basis="X_pca", random_state=0)
    gc.collect()

    # Cluster at resolution 0.6, same as baseline, single-threaded for reproducibility
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)
    sc.tl.leiden(adata, resolution=0.6, key_added="leiden_0.6",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    n_clusters = adata.obs["leiden_0.6"].nunique()
    print(f"\nFinal: {adata.n_obs} cells, {n_clusters} clusters at resolution 0.6")

    return adata, n_before, n_after, n_clusters

print("Pipeline function ready")

Pipeline function ready


In [4]:
# ----------------------------
# Cell 3 — Run at LOOSER threshold (n_genes > 300, vs baseline > 500)
# ----------------------------
adata_loose, n_before_loose, n_after_loose, n_clusters_loose = run_pipeline_at_threshold(
    300, "LOOSER than baseline"
)
gc.collect()


RUNNING PIPELINE AT n_genes_by_counts > 300 (LOOSER than baseline)

Cell filtering: 100064 -> 98270 cells (1794 removed, threshold n_genes>300)
Doublets removed: 239 (final: 98031 cells)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-22 14:47:23,358 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-22 14:48:15,248 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-22 14:48:16,579 - harmonypy - INFO - Iteration 1 of 10
2026-07-22 14:49:50,097 - harmonypy - INFO - Iteration 2 of 10
2026-07-22 14:51:20,767 - harmonypy - INFO - Iteration 3 of 10
2026-07-22 14:52:52,709 - harmonypy - INFO - Iteration 4 of 10
2026-07-22 14:54:24,310 - harmonypy - INFO - Iteration 5 of 10
2026-07-22 14:55:55,065 - harmonypy - INFO - Converged after 5 iterations
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Final: 98031 cells, 25 clusters at resolution 0.6


80824

In [12]:
# ----------------------------
# Cell 4 — Run at STRICTER threshold (n_genes > 750, vs baseline > 500)
# ----------------------------
adata_strict, n_before_strict, n_after_strict, n_clusters_strict = run_pipeline_at_threshold(
    750, "STRICTER than baseline"
)
gc.collect()


RUNNING PIPELINE AT n_genes_by_counts > 750 (STRICTER than baseline)

Cell filtering: 100064 -> 78050 cells (22014 removed, threshold n_genes>750)
Doublets removed: 174 (final: 77876 cells)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-14 15:34:49,397 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-14 15:35:26,099 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-14 15:35:26,844 - harmonypy - INFO - Iteration 1 of 10
2026-07-14 15:36:27,368 - harmonypy - INFO - Iteration 2 of 10
2026-07-14 15:37:28,735 - harmonypy - INFO - Iteration 3 of 10
2026-07-14 15:38:25,806 - harmonypy - INFO - Iteration 4 of 10
2026-07-14 15:39:28,867 - harmonypy - INFO - Iteration 5 of 10
2026-07-14 15:40:27,311 - harmonypy - INFO - Converged after 5 iterations



Final: 77876 cells, 32 clusters at resolution 0.6


1805

In [13]:
# ----------------------------
# Cell 5 — Compare cell counts and cluster counts against baseline
# Baseline (n_genes>500, already validated): 91,425 cells, 28 clusters
# ----------------------------
comparison = pd.DataFrame({
    "Threshold": ["Loose (>300)", "Baseline (>500)", "Strict (>750)"],
    "Cells retained": [n_after_loose, 91425, adata_strict.n_obs],
    "Clusters (res 0.6)": [n_clusters_loose, 28, n_clusters_strict],
})
print(comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR / "GSE176078_QC_threshold_sensitivity_summary.csv", index=False)

print("\n>>> Interpretation: a robust pipeline should show cell/cluster counts")
print(">>> change gradually and sensibly with threshold (looser = more cells,")
print(">>> similar or slightly more clusters; stricter = fewer cells, similar or")
print(">>> slightly fewer clusters) rather than wildly different structure.")

      Threshold  Cells retained  Clusters (res 0.6)
   Loose (>300)           98270                  25
Baseline (>500)           91425                  28
  Strict (>750)           77876                  32

>>> Interpretation: a robust pipeline should show cell/cluster counts
>>> change gradually and sensibly with threshold (looser = more cells,
>>> similar or slightly more clusters; stricter = fewer cells, similar or
>>> slightly fewer clusters) rather than wildly different structure.


In [14]:
# ----------------------------
# Cell 6 — Strict-threshold check + full comparison
# ----------------------------
prop_strict = check_memory_t_pattern(adata_strict, "Strict (>750)")
if prop_strict is not None:
    prop_strict.to_csv(RESULTS_DIR / "GSE176078_strict_threshold_memoryT_check.csv")

# Reload loose-threshold result (object was freed earlier for memory)
prop_loose_reloaded = pd.read_csv(
    RESULTS_DIR / "GSE176078_loose_threshold_memoryT_check.csv", index_col=0
)
bcell_strict_results = check_bcell_tnf_pathway(adata_strict, "Strict (>750)")
if bcell_strict_results:
    pd.DataFrame(bcell_strict_results).T.to_csv(RESULTS_DIR / "GSE176078_strict_Bcell_TNF_check.csv")

print("\n=== Comparison across all three thresholds ===")
print("Loose (>300):")
print(prop_loose_reloaded)
print("\nStrict (>750):")
print(prop_strict)
print("\n>>> Baseline (validated, n_genes>500): Memory T cells credibly elevated")
print(">>> in HER2+ vs both ER+ and TNBC (scCODA). Check above: does HER2+ show")
print(">>> the highest memory-T-like %% at both alternative thresholds too?")

  Strict (>750): memory-T-like cell %% by subtype:
    {'ER+': 2.138686383581522, 'HER2+': 8.492874622139054, 'TNBC': 5.813522075401695}

=== Comparison across all three thresholds ===
Loose (>300):
         memory_t_like
subtype               
ER+           3.512762
HER2+         7.402963
TNBC          5.223575

Strict (>750):
subtype
ER+      2.138686
HER2+    8.492875
TNBC     5.813522
Name: memory_t_like, dtype: float64

>>> Baseline (validated, n_genes>500): Memory T cells credibly elevated
>>> in HER2+ vs both ER+ and TNBC (scCODA). Check above: does HER2+ show
>>> the highest memory-T-like %% at both alternative thresholds too?
